# BACKGEOUND SUBTRCTION

In [3]:
import cv2
import numpy as np

# Load video
cap = cv2.VideoCapture("mus.mp4")  # Replace with your video file

# Read the first frame as the background reference
ret, background = cap.read()
if not ret:
    raise ValueError("Error loading video. Check the file path.")

# Convert background to grayscale
bg_gray = cv2.cvtColor(background, cv2.COLOR_BGR2GRAY)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break  # Exit loop if video ends

    # Convert current frame to grayscale
    frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Compute absolute difference
    fg_mask = cv2.absdiff(bg_gray, frame_gray)

    # Apply threshold to detect moving objects
    _, fg_mask = cv2.threshold(fg_mask, 50, 255, cv2.THRESH_BINARY)

    # Display results
    cv2.imshow("Original Frame", frame)
    cv2.imshow("Foreground Mask", fg_mask)

    # Press 'q' to exit the loop
    if cv2.waitKey(30) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


# BACKGROUND AVERAGING

In [10]:
import cv2
import numpy as np

# Load video
cap = cv2.VideoCapture("mus.mp4")  # Replace with your video file

# Initialize background model
avg = None

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break  # Exit loop if video ends

    # Convert to grayscale
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Initialize average background model
    if avg is None:
        avg = np.float32(gray)

    # Compute running average of background
    cv2.accumulateWeighted(gray, avg, 0.01)  # Adjust 0.01 for faster/slower adaptation

    # Convert the accumulated average to uint8 format
    bg_avg = cv2.convertScaleAbs(avg)

    # Compute difference between current frame and background
    fg_mask = cv2.absdiff(gray, bg_avg)

    # Apply threshold to detect moving objects
    _, fg_mask = cv2.threshold(fg_mask, 50, 255, cv2.THRESH_BINARY)

    # Display results
    cv2.imshow("Live Frame", frame)
    cv2.imshow("Background Average", bg_avg)
    cv2.imshow("Foreground Mask", fg_mask)

    # Press 'q' to exit
    if cv2.waitKey(30) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


# SCENE CHANGE DETECTION

In [32]:
import cv2
import numpy as np
from skimage.metrics import structural_similarity as ssim

# Load video
cap = cv2.VideoCapture("mus.mp4")  # Replace with your video file
ret, prev_frame = cap.read()

if not ret:
    raise ValueError("Failed to capture video.")

# Convert first frame to grayscale
prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Define a region of interest (ROI) for better detection (center of frame)
    h, w = gray.shape
    roi_size = (h // 2, w // 2)  # Use half of the frame size
    x_start, y_start = w // 4, h // 4  # Center the ROI

    roi_prev = prev_gray[y_start:y_start + roi_size[0], x_start:x_start + roi_size[1]]
    roi_curr = gray[y_start:y_start + roi_size[0], x_start:x_start + roi_size[1]]

    # Compute Structural Similarity Index (SSIM)
    similarity = ssim(roi_prev, roi_curr)

    # Debug: Print similarity score
    print(f"SSIM Score: {similarity}")

    # Detect scene change if similarity drops significantly
    if similarity < 0.8:  # Adjust threshold based on printed values
        print("⚠️ Scene Change Detected!")
        cv2.rectangle(frame, (x_start, y_start), (x_start + roi_size[1], y_start + roi_size[0]), (0, 0, 255), 5)  # Red box around ROI

    prev_gray = gray.copy()

    cv2.imshow("Live Frame", frame)

    if cv2.waitKey(30) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


SSIM Score: 0.6007577559302487
⚠️ Scene Change Detected!
SSIM Score: 0.7520213739492195
⚠️ Scene Change Detected!
SSIM Score: 0.7755945440604622
⚠️ Scene Change Detected!
SSIM Score: 0.7849551538650263
⚠️ Scene Change Detected!
SSIM Score: 0.8347960342324126
SSIM Score: 0.8769446113860667
SSIM Score: 0.9300159218254536
SSIM Score: 0.8809261964266436
SSIM Score: 0.9066185464452636
SSIM Score: 0.9116274557855819
SSIM Score: 0.9105422664751953
SSIM Score: 0.883369998422254
SSIM Score: 0.8883857317689984
SSIM Score: 0.8773243493641767
SSIM Score: 0.8982922272943421
SSIM Score: 0.9178442162835879
SSIM Score: 0.9049967237298692
SSIM Score: 0.9256008716254825
SSIM Score: 0.9199151096918191
SSIM Score: 0.9033716119300212
SSIM Score: 0.9038117874305168
SSIM Score: 0.9291825883845785
SSIM Score: 0.9220864980733816
SSIM Score: 0.922467499783145
SSIM Score: 0.9283301856285011
SSIM Score: 0.9313288563820612
SSIM Score: 0.9009831917556359
SSIM Score: 0.897275714995017
SSIM Score: 0.8941830698024383


In [38]:
import cv2
import numpy as np

# Load video
cap = cv2.VideoCapture("mus.mp4")  # Replace with your video file
ret, prev_frame = cap.read()

if not ret:
    raise ValueError("Failed to capture video.")

# Convert first frame to grayscale
prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Compute absolute difference
    diff = cv2.absdiff(prev_gray, gray)

    # Threshold to highlight significant changes
    _, diff_thresh = cv2.threshold(diff, 30, 255, cv2.THRESH_BINARY)

    # Calculate the percentage of changed pixels
    change_percentage = (np.sum(diff_thresh > 0) / diff_thresh.size) * 100

    # Print if significant change occurs
    if change_percentage > 10:  # Adjust threshold as needed
        print(f"⚠️ Scene Change Detected! ({change_percentage:.2f}% pixels changed)")

    prev_gray = gray.copy()

    cv2.imshow("Live Frame", frame)
    cv2.imshow("Difference", diff_thresh)

    if cv2.waitKey(30) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


⚠️ Scene Change Detected! (18.04% pixels changed)
⚠️ Scene Change Detected! (13.66% pixels changed)
⚠️ Scene Change Detected! (11.27% pixels changed)
⚠️ Scene Change Detected! (10.51% pixels changed)
⚠️ Scene Change Detected! (29.53% pixels changed)
⚠️ Scene Change Detected! (46.93% pixels changed)
⚠️ Scene Change Detected! (34.15% pixels changed)
⚠️ Scene Change Detected! (28.81% pixels changed)
⚠️ Scene Change Detected! (54.70% pixels changed)
⚠️ Scene Change Detected! (55.51% pixels changed)
⚠️ Scene Change Detected! (53.02% pixels changed)
⚠️ Scene Change Detected! (51.17% pixels changed)
⚠️ Scene Change Detected! (33.21% pixels changed)
⚠️ Scene Change Detected! (12.81% pixels changed)
⚠️ Scene Change Detected! (18.63% pixels changed)
⚠️ Scene Change Detected! (18.52% pixels changed)
